# 04 · Reproduce the key figures

Two of the project's headline figures, from the pre-computed vectors:
(1) per-trait transfer heatmaps, (2) the shared-variance bar chart (IV vs CAA).

In [ ]:
import torch
from pathlib import Path
from persona_steering.utils import parse_persona_trait_from_stem

def load_vector_dir(vectors_dir, layer=22):
    """Load all steering vectors from a dir of .pt files at one layer.

    Robust to two on-disk formats:
      * dict payload  {'vector': Tensor[L, D], 'persona':..., 'trait':...}
      * a raw Tensor[L, D]
    Returns {(persona, trait): Tensor[D]} for the requested layer.
    """
    vectors_dir = Path(vectors_dir)
    out = {}
    for pt in sorted(vectors_dir.glob('*.pt')):
        obj = torch.load(pt, map_location='cpu', weights_only=False)
        vec = obj['vector'] if isinstance(obj, dict) and 'vector' in obj else obj
        vec = vec.float()
        persona, trait = parse_persona_trait_from_stem(pt.stem)
        if persona and trait and layer < vec.shape[0]:
            out[(persona, trait)] = vec[layer]
    return out

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np, matplotlib.pyplot as plt
from persona_steering.config import Trait
from persona_steering.utils import VectorShim
from persona_steering.analysis import build_per_trait_transfer, decompose_shared_specific

DATA, LAYER = Path('hf_data'), 22
flat = load_vector_dir(DATA / 'vectors', layer=LAYER)
personas = sorted({p for (p, _) in flat})
traits = [Trait(t) for t in sorted({t for (_, t) in flat})]
nested = defaultdict(lambda: defaultdict(dict))
for (p, t), v in flat.items():
    nested[p][Trait(t)][LAYER] = VectorShim(v, p, Trait(t), LAYER)

### Figure 1 — per-trait cross-persona heatmaps

In [ ]:
ncol = 4; nrow = int(np.ceil(len(traits) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow))
for ax, t in zip(axes.flat, traits):
    pm = build_per_trait_transfer(nested, personas, t, LAYER)
    im = ax.imshow(pm, vmin=0, vmax=1, cmap='RdYlBu_r')
    ax.set_title(t.value, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(traits):]:
    ax.axis('off')
fig.suptitle(f'Per-trait cross-persona cosine similarity (IV, layer {LAYER})')
fig.colorbar(im, ax=axes, shrink=0.6, label='cosine'); plt.show()

### Figure 2 — shared variance, IV vs CAA

In [ ]:
def sv(vectors_dir):
    f = load_vector_dir(vectors_dir, layer=LAYER)
    ts = sorted({t for (_, t) in f})
    return {t: decompose_shared_specific({p: VectorShim(v, p, t, LAYER)
            for (p, tt), v in f.items() if tt == t}).variance_explained for t in ts}

iv_tab, caa_tab = sv(DATA / 'vectors'), sv(DATA / 'caa_vectors')
order = sorted(iv_tab, key=lambda k: -iv_tab[k])
x = np.arange(len(order)); w = 0.4
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, [iv_tab[t]*100 for t in order], w, label='IV')
ax.bar(x + w/2, [caa_tab.get(t, np.nan)*100 for t in order], w, label='CAA')
ax.axhline(80, ls='--', c='grey', lw=1)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('shared variance (%)'); ax.set_title('Shared variance by trait'); ax.legend()
plt.tight_layout(); plt.show()

That's the onboarding tour. From here:
- Read [`docs/experiments.md`](../docs/experiments.md) for the extended experiments (E2–E7).
- Pick up a [good first issue](https://github.com/jacobdaviescam/steering_across_personas/issues).
- See [`CONTRIBUTING.md`](../CONTRIBUTING.md) for the branch + PR workflow.